# 2. Feature Binning

This notebook bins only the `device_trust_score` feature after outlier handling and before class imbalance handling.

## Pipeline
1. Load the outlier-handled dataset
2. Select `device_trust_score` for binning
3. Create `device_trust_score_binned`
4. Drop the original `device_trust_score` column
5. Verify and save the transformed dataset

> Binning before SMOTE helps convert continuous values into stable categories that can be handled cleanly in the imbalance notebook.

## 1. Imports & Setup

In [40]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Imports successful.')

Imports successful.


## 2. Load Processed Data

In [41]:
INPUT_PATH = 'C:/Users/2021ICTS28/Desktop/end-to-end-credit-card-fraud-detection-system/dataset/processed/credit_card_fraud_outliers_handled.csv'

df = pd.read_csv(INPUT_PATH)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

Loaded: 10,000 rows x 10 columns


,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
0,1,84.4700,22,Electronics,0,0,66,3,40,0
1,2,529.8425,3,Travel,1,0,87,1,64,0
2,3,237.0100,17,Grocery,0,0,49,1,61,0
3,4,164.3300,4,Grocery,0,1,72,3,34,0
4,5,30.5300,15,Food,0,0,79,0,44,0


## 3. Select Column to Bin

In [42]:
BINNING_COL = 'device_trust_score'
NEW_BINNED_COL = 'device_trust_score_binned'

if BINNING_COL not in df.columns:
    raise KeyError(f"Column '{BINNING_COL}' not found in dataset.")

print(f"Column selected for binning: {BINNING_COL}")

Column selected for binning: device_trust_score


## 4. Apply Feature Binning

In [43]:
def map_device_trust_sector(score):
    if pd.isna(score):
        return np.nan
    elif score < 25:
        return 'Poor'
    elif score < 50:
        return 'Fair'
    elif score < 75:
        return 'Good'
    else:
        return 'Excellent'

df_binned = df.copy()
df_binned[NEW_BINNED_COL] = df_binned[BINNING_COL].apply(map_device_trust_sector)
df_binned.drop(columns=[BINNING_COL], inplace=True)

print(f"Binned column '{BINNING_COL}' into labeled column '{NEW_BINNED_COL}' using fixed ranges.")
print('Ranges: <25 Poor | 25-49 Fair | 50-74 Good | >=75 Excellent')
df_binned[[NEW_BINNED_COL]].head()

Binned column 'device_trust_score' into labeled column 'device_trust_score_binned' using fixed ranges.
Ranges: <25 Poor | 25-49 Fair | 50-74 Good | >=75 Excellent


,device_trust_score_binned
0,Good
1,Excellent
2,Fair
3,Good
4,Excellent


## 5. Verify the Transformation

In [44]:
print('Dtype summary after binning:')
print(df_binned.dtypes.value_counts())

print()
print(f"Value counts for {NEW_BINNED_COL}:")
print(df_binned[NEW_BINNED_COL].value_counts(dropna=False).head(10))

print()
print(f"Column '{BINNING_COL}' present after transform? {BINNING_COL in df_binned.columns}")

Dtype summary after binning:
int64      7
str        2
float64    1
Name: count, dtype: int64

Value counts for device_trust_score_binned:
device_trust_score_binned
Fair         3359
Good         3358
Excellent    3283
Name: count, dtype: int64

Column 'device_trust_score' present after transform? False


## 6. Save Outputs

In [45]:
OUTPUT_PATH = 'C:/Users/2021ICTS28/Desktop/end-to-end-credit-card-fraud-detection-system/dataset/processed/credit_card_fraud_binned.csv'

output_dir = os.path.dirname(OUTPUT_PATH)
os.makedirs(output_dir, exist_ok=True)
df_binned.to_csv(OUTPUT_PATH, index=False)

print(f'Saved binned dataset to: {OUTPUT_PATH}')
print(f'Rows: {df_binned.shape[0]:,} | Columns: {df_binned.shape[1]:,}')

Saved binned dataset to: C:/Users/2021ICTS28/Desktop/end-to-end-credit-card-fraud-detection-system/dataset/processed/credit_card_fraud_binned.csv
Rows: 10,000 | Columns: 10
